# Image → 3D Batch (Colab)

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run each cell below in order (▶). Allow Google Drive access when asked.
3. The last cell gives you a link to the app. Finished models are saved to `MyDrive/img23d`.

First run takes several minutes (installs + ~1.7 GB model download).

In [ ]:
# 1) Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then re-run.'

In [ ]:
# 2) Save outputs to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/img23d', exist_ok=True)

In [ ]:
# 3) Write the app files
import os, json
BUNDLE = json.loads('{"app.py": "\\"\\"\\"Batch 2D image -> 3D model web app. Run: uvicorn app:app --port 8000\\"\\"\\"\\nimport io\\nimport os\\nimport queue\\nimport shutil\\nimport threading\\nimport uuid\\nimport zipfile\\nfrom pathlib import Path\\n\\nfrom fastapi import FastAPI, File, HTTPException, UploadFile\\nfrom fastapi.responses import FileResponse, StreamingResponse\\nfrom fastapi.staticfiles import StaticFiles\\n\\nfrom providers import get_provider\\n\\nDATA = Path(os.environ.get(\\"DATA_DIR\\", Path(__file__).parent / \\"data\\"))\\nIN_DIR, OUT_DIR = DATA / \\"inputs\\", DATA / \\"outputs\\"\\nfor d in (IN_DIR, OUT_DIR):\\n    d.mkdir(parents=True, exist_ok=True)\\n\\nALLOWED = {\\".png\\", \\".jpg\\", \\".jpeg\\", \\".webp\\", \\".bmp\\"}\\nMAX_BYTES = 25 * 1024 * 1024\\n\\napp = FastAPI(title=\\"img23d\\")\\njobs: dict[str, dict] = {}\\norder: list[str] = []\\nq: \\"queue.Queue[str]\\" = queue.Queue()\\nprovider = None\\n\\n\\ndef worker():\\n    global provider\\n    while True:\\n        job_id = q.get()\\n        job = jobs.get(job_id)\\n        if not job or job[\\"status\\"] == \\"cancelled\\":\\n            continue\\n        try:\\n            if provider is None:\\n                job[\\"status\\"] = \\"loading model\\"\\n                provider = get_provider()\\n            job[\\"status\\"] = \\"processing\\"\\n            provider.generate(job[\\"input\\"], job[\\"output\\"])\\n            job[\\"status\\"] = \\"done\\"\\n        except Exception as e:  # keep the batch going\\n            job[\\"status\\"] = \\"failed\\"\\n            job[\\"error\\"] = f\\"{type(e).__name__}: {e}\\"\\n        finally:\\n            q.task_done()\\n\\n\\nthreading.Thread(target=worker, daemon=True).start()\\n\\n\\ndef public(job):\\n    return {k: job[k] for k in (\\"id\\", \\"name\\", \\"status\\", \\"error\\")}\\n\\n\\n@app.post(\\"/api/upload\\")\\nasync def upload(files: list[UploadFile] = File(...)):\\n    created = []\\n    for f in files:\\n        ext = Path(f.filename or \\"\\").suffix.lower()\\n        if ext not in ALLOWED:\\n            created.append({\\"name\\": f.filename, \\"status\\": \\"failed\\", \\"error\\": \\"unsupported file type\\", \\"id\\": uuid.uuid4().hex})\\n            continue\\n        data = await f.read()\\n        if len(data) > MAX_BYTES:\\n            created.append({\\"name\\": f.filename, \\"status\\": \\"failed\\", \\"error\\": \\"file too large (25MB max)\\", \\"id\\": uuid.uuid4().hex})\\n            continue\\n        job_id = uuid.uuid4().hex\\n        inp = IN_DIR / f\\"{job_id}{ext}\\"\\n        inp.write_bytes(data)\\n        job = {\\"id\\": job_id, \\"name\\": Path(f.filename).name, \\"status\\": \\"queued\\", \\"error\\": None,\\n               \\"input\\": str(inp), \\"output\\": str(OUT_DIR / f\\"{job_id}.glb\\")}\\n        jobs[job_id] = job\\n        order.append(job_id)\\n        q.put(job_id)\\n        created.append(public(job))\\n    return created\\n\\n\\n@app.get(\\"/api/jobs\\")\\ndef list_jobs():\\n    return [public(jobs[i]) for i in order if i in jobs]\\n\\n\\ndef get_job(job_id):\\n    if job_id not in jobs:\\n        raise HTTPException(404, \\"no such job\\")\\n    return jobs[job_id]\\n\\n\\n@app.get(\\"/api/jobs/{job_id}/model.glb\\")\\ndef model(job_id: str):\\n    job = get_job(job_id)\\n    if job[\\"status\\"] != \\"done\\":\\n        raise HTTPException(409, \\"not ready\\")\\n    return FileResponse(job[\\"output\\"], media_type=\\"model/gltf-binary\\",\\n                        filename=Path(job[\\"name\\"]).stem + \\".glb\\")\\n\\n\\n@app.get(\\"/api/jobs/{job_id}/input\\")\\ndef job_input(job_id: str):\\n    return FileResponse(get_job(job_id)[\\"input\\"])\\n\\n\\n@app.post(\\"/api/jobs/{job_id}/retry\\")\\ndef retry(job_id: str):\\n    job = get_job(job_id)\\n    if job[\\"status\\"] == \\"failed\\" and Path(job[\\"input\\"]).exists():\\n        job.update(status=\\"queued\\", error=None)\\n        q.put(job_id)\\n    return public(job)\\n\\n\\n@app.delete(\\"/api/jobs/{job_id}\\")\\ndef delete(job_id: str):\\n    job = get_job(job_id)\\n    job[\\"status\\"] = \\"cancelled\\"\\n    for p in (job[\\"input\\"], job[\\"output\\"]):\\n        Path(p).unlink(missing_ok=True)\\n    jobs.pop(job_id, None)\\n    if job_id in order:\\n        order.remove(job_id)\\n    return {\\"ok\\": True}\\n\\n\\n@app.get(\\"/api/download.zip\\")\\ndef download_all():\\n    buf = io.BytesIO()\\n    seen = {}\\n    with zipfile.ZipFile(buf, \\"w\\", zipfile.ZIP_STORED) as z:\\n        for i in order:\\n            j = jobs.get(i)\\n            if j and j[\\"status\\"] == \\"done\\":\\n                stem = Path(j[\\"name\\"]).stem\\n                seen[stem] = seen.get(stem, 0) + 1\\n                name = f\\"{stem}.glb\\" if seen[stem] == 1 else f\\"{stem}_{seen[stem]}.glb\\"\\n                z.write(j[\\"output\\"], name)\\n    buf.seek(0)\\n    return StreamingResponse(buf, media_type=\\"application/zip\\",\\n                             headers={\\"Content-Disposition\\": \'attachment; filename=\\"models.zip\\"\'})\\n\\n\\napp.mount(\\"/\\", StaticFiles(directory=Path(__file__).parent / \\"static\\", html=True), name=\\"static\\")\\n", "requirements.txt": "fastapi\\nuvicorn[standard]\\npython-multipart\\ntrimesh\\nnumpy\\npillow\\n", "providers/__init__.py": "import os\\n\\n\\ndef get_provider(name: str | None = None):\\n    \\"\\"\\"Pick the image->3D backend. Add new ones here (e.g. a hosted API).\\"\\"\\"\\n    name = (name or os.environ.get(\\"PROVIDER\\", \\"triposr\\")).lower()\\n    if name == \\"mock\\":\\n        from .mock import MockProvider\\n        return MockProvider()\\n    if name == \\"triposr\\":\\n        from .triposr import TripoSRProvider\\n        return TripoSRProvider()\\n    raise ValueError(f\\"Unknown provider: {name}\\")\\n", "providers/triposr.py": "\\"\\"\\"TripoSR (open source, runs locally, free). Needs the TripoSR repo on PYTHONPATH\\n(setup script clones it to ./TripoSR).\\"\\"\\"\\nimport os\\nimport sys\\nimport threading\\n\\nHERE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\\nsys.path.insert(0, os.path.join(HERE, \\"TripoSR\\"))\\n\\n\\nclass TripoSRProvider:\\n    name = \\"triposr\\"\\n\\n    def __init__(self):\\n        import torch\\n        import rembg\\n        from tsr.system import TSR\\n\\n        self.torch = torch\\n        self.device = \\"cuda:0\\" if torch.cuda.is_available() else \\"cpu\\"\\n        self.resolution = int(os.environ.get(\\"MESH_RESOLUTION\\", \\"256\\"))\\n        self.lock = threading.Lock()\\n        print(f\\"[triposr] loading model on {self.device} ...\\", flush=True)\\n        self.model = TSR.from_pretrained(\\n            \\"stabilityai/TripoSR\\", config_name=\\"config.yaml\\", weight_name=\\"model.ckpt\\"\\n        )\\n        self.model.renderer.set_chunk_size(8192)\\n        self.model.to(self.device)\\n        self.rembg_session = rembg.new_session()\\n        print(\\"[triposr] ready\\", flush=True)\\n\\n    def generate(self, image_path: str, out_path: str) -> None:\\n        import numpy as np\\n        from PIL import Image\\n        from tsr.utils import remove_background, resize_foreground\\n\\n        with self.lock:\\n            image = Image.open(image_path).convert(\\"RGB\\")\\n            image = remove_background(image, self.rembg_session)\\n            image = resize_foreground(image, 0.85)\\n            arr = np.array(image).astype(np.float32) / 255.0\\n            arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5\\n            image = Image.fromarray((arr * 255.0).astype(np.uint8))\\n            with self.torch.no_grad():\\n                codes = self.model(image, device=self.device)\\n            mesh = self.model.extract_mesh(codes, True, resolution=self.resolution)[0]\\n            mesh.export(out_path)\\n            if self.device.startswith(\\"cuda\\"):\\n                self.torch.cuda.empty_cache()\\n", "providers/mock.py": "\\"\\"\\"Fake provider for testing the app without a GPU: makes a textured-colour sphere/box.\\"\\"\\"\\nimport time\\nimport numpy as np\\nimport trimesh\\nfrom PIL import Image\\n\\n\\nclass MockProvider:\\n    name = \\"mock\\"\\n\\n    def generate(self, image_path: str, out_path: str) -> None:\\n        img = Image.open(image_path).convert(\\"RGB\\").resize((8, 8))\\n        color = np.array(img).reshape(-1, 3).mean(axis=0).astype(np.uint8)\\n        mesh = trimesh.creation.icosphere(subdivisions=3, radius=0.5)\\n        mesh.visual.vertex_colors = np.tile(np.append(color, 255), (len(mesh.vertices), 1))\\n        time.sleep(1)\\n        mesh.export(out_path)\\n", "static/index.html": "<!doctype html>\\n<html lang=\\"en\\">\\n<head>\\n<meta charset=\\"utf-8\\">\\n<meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1\\">\\n<title>Image to 3D Batch</title>\\n<script type=\\"module\\" src=\\"https://cdnjs.cloudflare.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js\\"></script>\\n<style>\\n  :root { --bg:#f6f7f9; --card:#fff; --text:#1c1f24; --muted:#6b7280; --accent:#4f46e5; --border:#e3e5e9; --bad:#dc2626; --ok:#16a34a; }\\n  @media (prefers-color-scheme: dark) { :root { --bg:#14161a; --card:#1d2025; --text:#e8eaed; --muted:#9aa0a8; --border:#2c3037; } }\\n  * { box-sizing:border-box } body { margin:0; font:15px system-ui,sans-serif; background:var(--bg); color:var(--text); }\\n  main { max-width:1100px; margin:0 auto; padding:24px 16px 64px; }\\n  h1 { font-size:22px; margin:0 0 4px } p.sub { color:var(--muted); margin:0 0 20px }\\n  #drop { border:2px dashed var(--border); border-radius:12px; padding:36px 16px; text-align:center; background:var(--card); cursor:pointer; }\\n  #drop.over { border-color:var(--accent); } #drop b { color:var(--accent) }\\n  .bar { display:flex; gap:12px; align-items:center; margin:20px 0 12px; flex-wrap:wrap }\\n  .bar span { color:var(--muted); flex:1 }\\n  button { background:var(--accent); color:#fff; border:0; border-radius:8px; padding:8px 14px; font:inherit; cursor:pointer }\\n  button.ghost { background:transparent; color:var(--text); border:1px solid var(--border) }\\n  button:disabled { opacity:.45; cursor:default }\\n  #grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(240px,1fr)); gap:14px }\\n  .card { background:var(--card); border:1px solid var(--border); border-radius:12px; overflow:hidden; display:flex; flex-direction:column }\\n  .view { height:220px; background:var(--bg); display:flex; align-items:center; justify-content:center; position:relative }\\n  .view img { max-width:100%; max-height:100%; object-fit:contain; opacity:.6 }\\n  model-viewer { width:100%; height:100%; --poster-color:transparent }\\n  .meta { padding:10px 12px; display:flex; flex-direction:column; gap:6px }\\n  .name { font-weight:600; overflow:hidden; text-overflow:ellipsis; white-space:nowrap }\\n  .status { font-size:13px; color:var(--muted) } .status.done { color:var(--ok) } .status.failed { color:var(--bad) }\\n  .row { display:flex; gap:6px } .row button { padding:5px 10px; font-size:13px }\\n</style>\\n</head>\\n<body>\\n<main>\\n  <h1>Image \\u2192 3D Batch</h1>\\n  <p class=\\"sub\\">Drop as many images as you like. Each becomes a 3D model (.glb) you can preview and download.</p>\\n  <div id=\\"drop\\"><b>Click to choose</b> or drag images here\\n    <input id=\\"file\\" type=\\"file\\" accept=\\"image/*\\" multiple hidden></div>\\n  <div class=\\"bar\\"><span id=\\"summary\\">No images yet</span>\\n    <button class=\\"ghost\\" id=\\"clear\\">Clear finished</button>\\n    <button id=\\"zip\\" disabled>Download all (.zip)</button></div>\\n  <div id=\\"grid\\"></div>\\n</main>\\n<script>\\nconst $ = s => document.querySelector(s), drop = $(\'#drop\'), file = $(\'#file\'), grid = $(\'#grid\');\\ndrop.onclick = () => file.click();\\nfile.onchange = () => upload(file.files);\\n[\'dragover\',\'dragenter\'].forEach(e => drop.addEventListener(e, ev => { ev.preventDefault(); drop.classList.add(\'over\'); }));\\n[\'dragleave\',\'drop\'].forEach(e => drop.addEventListener(e, ev => { ev.preventDefault(); drop.classList.remove(\'over\'); }));\\ndrop.addEventListener(\'drop\', ev => upload(ev.dataTransfer.files));\\n$(\'#zip\').onclick = () => location.href = \'/api/download.zip\';\\n$(\'#clear\').onclick = async () => { for (const j of jobs) if (j.status === \'done\' || j.status === \'failed\') await fetch(\'/api/jobs/\' + j.id, {method:\'DELETE\'}); poll(); };\\n\\nasync function upload(files) {\\n  const BATCH = 10;                       // upload in chunks so big batches stay reliable\\n  const list = [...files];\\n  for (let i = 0; i < list.length; i += BATCH) {\\n    const fd = new FormData();\\n    list.slice(i, i + BATCH).forEach(f => fd.append(\'files\', f));\\n    await fetch(\'/api/upload\', {method:\'POST\', body:fd});\\n    poll();\\n  }\\n  file.value = \'\';\\n}\\n\\nlet jobs = [], built = {};\\nfunction el(tag, cls, text) { const e = document.createElement(tag); if (cls) e.className = cls; if (text) e.textContent = text; return e; }\\n\\nfunction render() {\\n  const ids = new Set(jobs.map(j => j.id));\\n  for (const id in built) if (!ids.has(id)) { built[id].card.remove(); delete built[id]; }\\n  for (const j of jobs) {\\n    let b = built[j.id];\\n    if (!b) {\\n      const card = el(\'div\',\'card\'), view = el(\'div\',\'view\'), meta = el(\'div\',\'meta\');\\n      const img = el(\'img\'); img.src = `/api/jobs/${j.id}/input`; view.append(img);\\n      const name = el(\'div\',\'name\', j.name), status = el(\'div\',\'status\'), row = el(\'div\',\'row\');\\n      meta.append(name, status, row); card.append(view, meta); grid.append(card);\\n      b = built[j.id] = {card, view, status, row, shown:null};\\n    }\\n    b.status.textContent = j.status + (j.error ? \': \' + j.error : \'\');\\n    b.status.className = \'status \' + j.status;\\n    if (b.shown !== j.status) {\\n      b.shown = j.status; b.row.replaceChildren();\\n      if (j.status === \'done\') {\\n        const mv = document.createElement(\'model-viewer\');\\n        mv.src = `/api/jobs/${j.id}/model.glb`; mv.setAttribute(\'camera-controls\',\'\'); mv.setAttribute(\'auto-rotate\',\'\');\\n        mv.setAttribute(\'shadow-intensity\',\'1\'); b.view.replaceChildren(mv);\\n        const dl = el(\'button\',\'\', \'Download .glb\'); dl.onclick = () => location.href = mv.src; b.row.append(dl);\\n      }\\n      if (j.status === \'failed\') { const r = el(\'button\',\'\',\'Retry\'); r.onclick = async () => { await fetch(`/api/jobs/${j.id}/retry`, {method:\'POST\'}); poll(); }; b.row.append(r); }\\n      if (j.status !== \'processing\' && j.status !== \'loading model\') {\\n        const d = el(\'button\',\'ghost\',\'Remove\'); d.onclick = async () => { await fetch(\'/api/jobs/\' + j.id, {method:\'DELETE\'}); poll(); }; b.row.append(d);\\n      }\\n    }\\n  }\\n  const done = jobs.filter(j => j.status === \'done\').length, failed = jobs.filter(j => j.status === \'failed\').length;\\n  $(\'#summary\').textContent = jobs.length ? `${done} of ${jobs.length} done` + (failed ? `, ${failed} failed` : \'\') : \'No images yet\';\\n  $(\'#zip\').disabled = !done;\\n}\\nasync function poll() { try { jobs = await (await fetch(\'/api/jobs\')).json(); render(); } catch (e) {} }\\nsetInterval(poll, 1500); poll();\\n</script>\\n</body>\\n</html>\\n"}')
for path, text in BUNDLE.items():
    full = os.path.join('/content/img23d', path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    open(full, 'w').write(text)
print('wrote', len(BUNDLE), 'files')

In [ ]:
# 4) Install dependencies (a few minutes)
%cd /content/img23d
!pip -q install -r requirements.txt
!test -d TripoSR || git clone -q https://github.com/VAST-AI-Research/TripoSR.git
!pip -q install omegaconf einops transformers "rembg[cpu]" huggingface-hub xatlas moderngl onnxruntime "imageio[ffmpeg]"
!pip -q install git+https://github.com/tatsy/torchmcubes.git

In [ ]:
# 5) Start the app and get your link
import subprocess, time, os
env = dict(os.environ, DATA_DIR='/content/drive/MyDrive/img23d', PROVIDER='triposr')
subprocess.Popen(['python','-m','uvicorn','app:app','--port','8000'], cwd='/content/img23d', env=env)
time.sleep(5)
from google.colab import output
output.serve_kernel_port_as_window(8000)
print('If no tab opened, allow pop-ups, or re-run this cell. Keep this notebook tab open while using the app.')